In [14]:
%load_ext autoreload
%autoreload 2

import json
import os
import yaml
from pathlib import Path
from dask.distributed import Client
import dask.dataframe as dd
import networkx as nx
import sys
import pandas as pd

import ipycytoscape
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import intervals
import pygtrie
import seaborn as sns
import gc

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
use_local=False
if not use_local:
    with open(f'/g/g91/pandey2/.dftracer/configuration.yaml', 'r') as file:
        dlp_yaml = yaml.safe_load(file)
        app_root = dlp_yaml["app"]
else:
    app_root = str(Path(os.getcwd()).parent.parent)
sys.path.insert(0, app_root)

import dfanalyzer
print(dfanalyzer.__file__)
from dfanalyzer.main import DFAnalyzer,get_dft_configuration,update_dft_configuration,setup_logging,setup_dask_cluster, reset_dask_cluster, get_dft_configuration
from dfanalyzer.graph_visualization.cytoscape import GraphFunctions, CytoGraph
from dfanalyzer.graph1 import DFGrepInterferencePartitionBased, DFGrepBurstiness, DFGrepWorkflow, DFGrepWorkflow1

if not use_local:
    dask_run_dir = os.path.join(app_root, "dfanalyzer", "dask", "run_dir")
    with open (os.path.join(dask_run_dir, f"scheduler_{os.getenv('USER')}.json"), "r") as f:
        dask_scheduler = json.load(f)["address"]
else:
    dask_scheduler = None

# App Name
# app_name = "montage-pegasus-2mass-2deg-4node" #cosmoflow cm1
app_name = "1000_genome_pegasus_node_16"
# app_name = "cm1"
# app_name = "deepspeed-dlio-step100"
# app_name = "deepspeed-dlio-scr-step100"
# app_name = "bert"
# app_name = "unet3d"
# app_name = "resnet50"
cp_dir = "/p/lustre3/pandey2/logs/Results_Checkpoint/dataflow/"+app_name+"/"
os.makedirs(cp_dir, exist_ok=True)

condition_fn = None #

if app_name == "cm1":
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/cm1/APP/node-32/v1/COMPACT/*.pfw.gz"
    # cp_dir = "/p/lustre3/pandey2/logs/results_checkpoint"

elif app_name == "deepspeed-dlio-step100":
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.6-develop/corona/megatron-deepspeed/dlio-step100/node-16/v1/COMPACT/*.pfw.gz"
    # filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.6-develop/corona/megatron-deepspeed/dlio-scr-step100/node-16/v1/COMPACT/*.pfw.gz"
    # cp_dir =  "/p/lustre3/pandey2/logs/results_checkpoint"

elif app_name == "deepspeed-dlio-scr-step100":
    # filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.6-develop/corona/megatron-deepspeed/dlio-step100/node-16/v1/COMPACT/*.pfw.gz"
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.6-develop/corona/megatron-deepspeed/dlio-scr-step100/node-16/v1/COMPACT/*.pfw.gz"
    # cp_dir =  "/p/lustre3/pandey2/logs/results_checkpoint"

elif app_name == "montage-mpi-2mass-7deg":
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/montage/mpi-2mass-7deg/node-16/v1/RAW/*.pfw.gz"
    # cp_dir =  "/p/lustre3/pandey2/logs/results_checkpoint"

elif app_name =="montage-pegasus-2mass-2deg-4node":
    # filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/montage/pegasus-2mass-2deg/node-4/v1/COMPACT/*.pfw.gz"
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/montage/pegasus-2mass-2deg/node-4/v1/RAW/*.pfw.gz"
    filename = "/p/lustre3/pandey2/logs/RAW_copy/montage-pegasus-2mass-2deg-4node/RAW/*.pfw.gz" # Raw Copied
    # cp_dir =  "/p/lustre3/pandey2/logs/results_checkpoint"

elif app_name == "resnet50":
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/resnet50/dlio-v100/node-4/v1/COMPACT/*.pfw.gz"
    # cp_dir =  "/p/lustre3/pandey2/logs/results_checkpoint"

elif app_name == "unet3d":
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/unet3d/dlio-v100/node-16/v2/COMPACT/*.pfw.gz"
    # cp_dir =  "/p/lustre3/pandey2/logs/results_checkpoint"

elif app_name == "bert":
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/bert/v100/node-16/v1/COMPACT/*.pfw.gz"
    # cp_dir =  "/p/lustre3/pandey2/logs/results_checkpoint"

elif app_name == "1000_genome_pegasus_node_16":
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/1000-genome/pegasus/node-16/v3/COMPACT/*.pfw.gz"
    # cp_dir =  "/p/lustre3/pandey2/logs/results_checkpoint"
    
else:
    raise Exception("Unknown App name")


# Configuration 4 update log file dlp -> df
conf = update_dft_configuration(dask_scheduler=dask_scheduler, verbose=True, debug=True,
                                log_file=f"./dft_{os.getenv('USER')}.log", rebuild_index=False, time_approximate=True, 
                                host_pattern=r'lassen(\d+)', time_granularity=1e6, skip_hostname=True, conditions=condition_fn)
conf = get_dft_configuration()


# Setup
setup_logging()
setup_dask_cluster()
reset_dask_cluster()

[INFO] [13:04:34] Initialized Client with 96 workers and link http://134.9.71.20:8787/status [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:769]


/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/__init__.py


[INFO] [13:04:38] Restarting all workers [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:761]


In [17]:
def all_mount_points():
    with open("/proc/mounts", "r") as file:
        mount_points = [line.split()[1] for line in file]
    with open("/usr/workspace/pandey2/lassen_mounts", "r") as file:
        mount_p = [line.split()[1] for line in file]
    return mount_points+mount_p

mount_points = all_mount_points()
trie = pygtrie.StringTrie(zip(mount_points, [True] * len(mount_points)))

def cols_function(json_object, current_dict, time_approximate,condition_fn,load_data):
    d = {}
    def find_mount_point(path,trie):
        mount_point = trie.longest_prefix(path)
        if mount_point:
            return mount_point.key
        return '/'.join(path.split('/', 3)[:3])

    if "M" == json_object["ph"] and "FH" == json_object["name"] and "args" in json_object and "name" in json_object["args"]:
        d["mount_point"] = find_mount_point(trie=load_data["mount_point"],path=json_object["args"]["name"])
    if "args" in json_object and "M" != json_object["ph"]:
        if "ret" in json_object["args"]:
            d["size"] = int(json_object["args"]["ret"]) 

    if "name" in json_object:
        if (json_object["name"] in ["fwrite", "write","pwrite"]):
            d["prod"] = 1
            d["cons"] = 0
        else:
            d["prod"] = 0
            d["cons"] = 1
    return d

load_cols = {'size': "int64[pyarrow]", 'prod':"uint16[pyarrow]", 'cons':"uint16[pyarrow]" }
load_cols_metadata = {"FH":{'mount_point':"string[pyarrow]" }}


In [18]:
analyzer = DFAnalyzer(filename,load_fn=cols_function, load_cols=load_cols, load_data={"mount_point":trie}, metadata_cols = load_cols_metadata)

[INFO] [13:04:46] Created index for 44 files [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:428]
[INFO] [13:04:46] Total size of all files are <dask.bag.core.Item object at 0x1554a25c9b50> bytes [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:430]
[INFO] [13:04:46] test debug [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:433]
[INFO] [13:04:48] Loading 26758 batches out of 44 files and has 438063442 lines overall [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:444]
[INFO] [13:20:33] Loaded events [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:511]
[INFO] [13:20:33] Loaded plots with slope threshold: 45 [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:517]


In [19]:
# df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
# df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
# df1['fhash'] = df1['fhash'].str.replace('.0', '', regex=False)  # Remove '.0'
# df1['hhash'] = df1['hhash'].str.replace('.0', '', regex=False)  # Remove '.0'
# df3 = analyzer.host_hash.reset_index()[['hash', 'name']] 
# df3 = df3.rename(columns={'hash':'hhash'}) # update needed for this specific dataset
# result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
# result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
# analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname','fhash','pid','tid','prod','cons']] 
# # analyze_df['id'] = analyze_df.index
# # analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")
# # data_calls = ['read', 'write', 'fread', 'fwrite', 'pread', 'pwrite']
# # data_df = analyze_df[analyze_df["name"].isin(data_calls)]
# # metadata_df = analyze_df[~analyze_df["name"].isin(data_calls)]

if app_name == 'cm1':
    df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
    df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
    df1['fhash'] = df1['fhash'].str.replace('.0', '', regex=False)  # Remove '.0'
    df1['hhash'] = df1['hhash'].str.replace('.0', '', regex=False)  # Remove '.0'
    df3 = analyzer.host_hash.reset_index()[['hash', 'name']] 
    df3 = df3.rename(columns={'hash':'hhash'}) # update needed for this specific dataset
    result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
    result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
    analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname','fhash','pid','tid','prod','cons']] 
    # analyze_df['id'] = analyze_df.index
    analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")
    # data_calls = ['read', 'write', 'fread', 'fwrite', 'pread', 'pwrite']
    # data_df = analyze_df[analyze_df["name"].isin(data_calls)]
    # metadata_df = analyze_df[~analyze_df["name"].isin(data_calls)]

if app_name in ["deepspeed-dlio-step100","deepspeed-dlio-scr-step100","resnet50","unet3d"]:
    df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
    df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
    df3 = analyzer.host_hash.reset_index()[['hhash', 'name']]
    result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
    result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
    analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname','fhash','pid','tid','prod','cons']] 
    analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")

elif app_name == "montage-pegasus-2mass-2deg-4node":
    df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
    df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
    df3 = analyzer.host_hash.reset_index()[['hash', 'name']] 
    df3 = df3.rename(columns={'hash':'hhash'})
    result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
    result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
    analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname','fhash','pid','tid','prod','cons']] 
    analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")

elif app_name in ["1000_genome_pegasus_node_16"]:
    df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
    df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
    df3 = analyzer.host_hash.reset_index()[['hhash', 'name']] 
    result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
    result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
    analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname','fhash','pid','tid','prod','cons']] 
    analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")


elif app_name == "bert":
    df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
    df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
    df1['fhash'] = df1['fhash'].str.replace('.0', '', regex=False)  # Remove '.0'
    df3 = analyzer.host_hash.reset_index()[['hash', 'name']] 
    df3 = df3.rename(columns={'hash':'hhash'}) # update needed for this specific dataset
    result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
    result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
    analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname','fhash','pid','tid','prod','cons']] 
    analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")
    

In [24]:
# has_shared_pids = (
#     analyze_df.groupby("pid")["hostname"].nunique() > 1
# ).any().compute()
# has_shared_pids

In [20]:
analyze_df.head()
# has_shared_pids = (
#     filtered.groupby("pid")["hostname"].nunique() > 1
# ).any().compute()

,name,cat,size,ts,te,dur,trange,mount_point,hostname,fhash,pid,tid,prod,cons
0,__lxstat64,POSIX,<NA>,5246621376,5246621379,3,5246,/usr/WS2,corona236,1.1501328197703913e+19,269269,269269,0,1
1,__lxstat64,POSIX,<NA>,5246621389,5246621393,4,5246,/usr/WS2,corona236,1.18455783226356e+19,269269,269269,0,1
2,__lxstat64,POSIX,<NA>,5246621403,5246621408,5,5246,/usr/WS2,corona236,8.839091513462778e+18,269269,269269,0,1
3,__lxstat64,POSIX,<NA>,5246621420,5246621434,14,5246,/usr/WS2,corona236,9.72234954274544e+17,269269,269269,0,1
4,__lxstat64,POSIX,<NA>,5246621446,5246621452,6,5246,/usr/WS2,corona236,1.260549135367404e+19,269269,269269,0,1


In [21]:
wf = DFGrepWorkflow(analyze_df)
# wf1 = DFGrepWorkflow1(analyze_df)

In [22]:
wf.select_events()
# wf1.select_events()

In [23]:
wf.selected_events.head()

,name,cat,size,ts,te,dur,trange,mount_point,hostname,fhash,pid,tid,prod,cons


In [24]:
lvl1_results = wf.get_wfGraph()
lvl2_results = wf.get_wfGraph(level=2)
lvl3_results = wf.get_wfGraph(level=3)


In [25]:
wf.write_graph_data(lvl1_results,cp_dir+"lvl1")
wf.write_graph_data(lvl2_results,cp_dir+"lvl2")
wf.write_graph_data(lvl3_results,cp_dir+"lvl3")
# lvl1_results.to_parquet(f'{cp_dir}lvl1',engine='pyarrow',write_index=False)

In [28]:
lvl2_results.compute()

KeyboardInterrupt: 

In [27]:
lvl3_results.query("prod > 0").compute()

,fhash,hostname,prod,cons


In [13]:
lvl2_results.query("prod > 0").compute()

,pid,fhash,prod,cons
354561,1785246,5501,26901,0


In [17]:
lvl1_results.groupby('hostname').count().compute()

,pid,fhash,ts
hostname,,,
corona275,16,16,16
corona189,17,17,17
corona277,16,16,16
corona278,16,16,16


In [42]:
lvl2_results.compute().query("prod > 0")

,pid,fhash,prod,cons


In [25]:
lvl1_results.head()

,hostname,pid,fhash,ts
0,corona240,2636091,10056,248071581
1,corona240,2636092,10139,80546151
2,corona240,2636076,10144,164166844
3,corona240,2636079,10202,164230161
4,corona240,2636095,10217,192189383


In [28]:
data=wf.selected_events.groupby(["fhash","hostname"])['prod','cons'].sum().reset_index().query("prod > 0  and cons > 0")
data.compute()

,fhash,hostname,prod,cons
0,10005,corona261,1,4
1,10023,corona253,1,4
2,10026,corona262,8,4
3,10046,corona261,1,4
4,10050,corona262,1,4
...,...,...,...,...
48762,9960,corona282,3,4
48763,9963,corona248,1,4
48764,9977,corona245,1,4
48765,9978,corona245,4,4


In [37]:

# Build valid composite keys
valid_pairs = (
    wf.selected_events
      .groupby(["fhash", "hostname"])[["prod", "cons"]]
      .sum()
      .reset_index()
      .query("prod > 0 and cons > 0")
      .assign(fhash_host=lambda d: d["fhash"].astype(str) + "_" + d["hostname"].astype(str))
      [["fhash_host"]]
)


In [45]:
wf.selected_events = wf.selected_events.assign(
    fhash_host = wf.selected_events["fhash"].astype(str) + "_" + wf.selected_events["hostname"].astype(str)
)


In [46]:
wf.selected_events.head()

,name,cat,size,ts,te,dur,trange,mount_point,hostname,fhash,pid,tid,prod,cons,fhash_host
0,fopen,STDIO,<NA>,118391,118698,307,0,/usr/tce,corona261,33084,1296180,1296180,0,1,33084_corona261
1,fread,STDIO,3107,118731,118742,11,0,/usr/tce,corona261,33084,1296180,1296180,0,1,33084_corona261
2,fread,STDIO,0,118779,118780,1,0,/usr/tce,corona261,33084,1296180,1296180,0,1,33084_corona261
3,fclose,STDIO,<NA>,118790,118793,3,0,/usr/tce,corona261,33084,1296180,1296180,0,1,33084_corona261
4,fopen,STDIO,<NA>,118805,119314,509,0,/usr/tce,corona261,33084,1296180,1296180,0,1,33084_corona261


In [10]:
# make composite key
se = wf.selected_events.assign(
    fhash_host = wf.selected_events["fhash"].astype(str) + "_" + wf.selected_events["hostname"].astype(str)
)

# get valid keys as numpy
valid_keys = (
    se.groupby("fhash_host")[["prod","cons"]].sum()
      .query("prod > 0 and cons > 0")
      .compute()
      .index.to_numpy()
)

# filter
filtered = se[se["fhash_host"].isin(set(valid_keys))]

In [11]:
f1 = filtered.groupby(["pid","fhash"])[["prod","cons"]].sum()

In [67]:
f1.reset_index().query(" not (prod > 0 and cons > 0)").head()

,pid,fhash,prod,cons
5,1296180,17674,0,2
8,1296180,21176,0,3
11,1296180,24089,0,6
17,1296180,34644,0,1
20,1296180,37927,0,1


In [68]:
filtered.head()

,name,cat,size,ts,te,dur,trange,mount_point,hostname,fhash,pid,tid,prod,cons,fhash_host
16,__xstat,POSIX,<NA>,120195,120206,11,0,/usr/tce,corona261,43457,1296180,1296180,0,1,43457_corona261
138,open,POSIX,7,141932,141943,11,0,/dev/shm,corona261,60672,1296180,1296180,0,1,60672_corona261
139,unlink,POSIX,<NA>,141954,141959,5,0,/dev/shm,corona261,60672,1296180,1296180,0,1,60672_corona261
990,__xstat,POSIX,<NA>,2125148,2125292,144,2,/sys,corona261,34644,1296180,1296180,0,1,34644_corona261
993,__xstat,POSIX,<NA>,2125709,2125895,186,2,/sys,corona261,62628,1296180,1296180,0,1,62628_corona261


In [12]:
has_shared_pids = (
    filtered.groupby("pid")["hostname"].nunique() > 1
).any().compute()


In [9]:
lvl1_results = wf.get_wfGraph()
lvl2_results = wf.get_wfGraph(level=2)
lvl3_results = wf.get_wfGraph(level=3)

In [13]:
has_shared_pids

np.False_

In [11]:
lvl1_results.to_parquet(f'{cp_dir}lvl1',engine='pyarrow',write_index=False)
lvl2_results.to_parquet(f'{cp_dir}lvl2',engine='pyarrow',write_index=False)
wf.build_and_save_graph_l3(lvl3_results,cp_dir)


Graph saved: 17 nodes, 32 edges written to '/p/lustre3/pandey2/logs/Results_Checkpoint/dataflow/1000_genome_pegasus_node_16/l3.csv'


In [18]:
# analyze_df['pid_x'] = analyze_df['pid_hpt']
# dfworkflow = DFGrepWorkflow(analyze_df, app_name = app_name, trace_path=filename)
# # find the number of times each file is prod/cons
# wf = dfworkflow.compute_workflow()
# # graph_df = dfworkflow.create_graph_df(wf.compute(),pid_map={})
# # temp_graph = graph_df.groupby(['src','dest'])['wt'].min().reset_index()

In [ ]:
analyze_df['pid_x'] = analyze_df['pid_hpt']
dfworkflow = DFGrepWorkflow(analyze_df, app_name = app_name, trace_path=filename)
# find the number of times each file is prod/cons
wf = dfworkflow.create_workflow()
graph_df = dfworkflow.create_graph_df(wf.compute(),pid_map={})
temp_graph = graph_df.groupby(['src','dest'])['wt'].min().reset_index()
temp_graph.to_csv(cp_dir+"/dataflow/"+str(app_name)+"/hpt.csv",header=True, index=False)

In [ ]:
analyze_df['pid_x'] = analyze_df['pid_hp']
dfworkflow = DFGrepWorkflow(analyze_df, app_name = app_name, trace_path=filename)
# find the number of times each file is prod/cons
wf = dfworkflow.create_workflow()
graph_df = dfworkflow.create_graph_df(wf.compute(),pid_map={})
temp_graph = graph_df.groupby(['src','dest'])['wt'].min().reset_index()
temp_graph.to_csv(cp_dir+"/dataflow/"+str(app_name)+"/hp.csv",header=True, index=False)

In [ ]:
analyze_df['pid_x'] = analyze_df['pid_h']
dfworkflow = DFGrepWorkflow(analyze_df, app_name = app_name, trace_path=filename)
# find the number of times each file is prod/cons
wf = dfworkflow.create_workflow()
graph_df = dfworkflow.create_graph_df(wf.compute(),pid_map={})
temp_graph = graph_df.groupby(['src','dest'])['wt'].min().reset_index()
temp_graph.to_csv(cp_dir+"/dataflow/"+str(app_name)+"/h.csv",header=True, index=False)

In [ ]:
# if app_name in ["deepspeed-dlio-step100","deepspeed-dlio-scr-step100","resnet50","unet3d"]:
#     df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
#     df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
#     df3 = analyzer.host_hash.reset_index()[['hhash', 'name']]
#     result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
#     result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
#     analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname']] 
#     analyze_df['id'] = analyze_df.index
#     analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")
#     data_calls = ['read', 'write', 'fread', 'fwrite', 'pread', 'pwrite']
#     data_df = analyze_df[analyze_df["name"].isin(data_calls)]
#     metadata_df = analyze_df[~analyze_df["name"].isin(data_calls)]

# elif app_name == "montage-pegasus-2mass-2deg-4node":
#     df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
#     df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
#     df3 = analyzer.host_hash.reset_index()[['hash', 'name']] 
#     df3 = df3.rename(columns={'hash':'hhash'})
#     result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
#     result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
#     analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname']] 
#     analyze_df['id'] = analyze_df.index
#     analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")
#     data_calls = ['read', 'write', 'fread', 'fwrite', 'pread', 'pwrite']
#     data_df = analyze_df[analyze_df["name"].isin(data_calls)]
#     metadata_df = analyze_df[~analyze_df["name"].isin(data_calls)]

# elif app_name in ["1000_genome_pegasus_node_16"]:
#     df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
#     df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
#     df3 = analyzer.host_hash.reset_index()[['hhash', 'name']] 
#     result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
#     result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
#     analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname']] 
#     analyze_df['id'] = analyze_df.index
#     analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")
#     data_calls = ['read', 'write', 'fread', 'fwrite', 'pread', 'pwrite']
#     data_df = analyze_df[analyze_df["name"].isin(data_calls)]
#     metadata_df = analyze_df[~analyze_df["name"].isin(data_calls)]


# elif app_name == "bert":
#     df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
#     df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
#     df1['fhash'] = df1['fhash'].str.replace('.0', '', regex=False)  # Remove '.0'
#     df3 = analyzer.host_hash.reset_index()[['hash', 'name']] 
#     df3 = df3.rename(columns={'hash':'hhash'}) # update needed for this specific dataset
#     result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
#     result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
#     analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname']] 
#     analyze_df['id'] = analyze_df.index
#     analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")
#     data_calls = ['read', 'write', 'fread', 'fwrite', 'pread', 'pwrite']
#     data_df = analyze_df[analyze_df["name"].isin(data_calls)]
#     metadata_df = analyze_df[~analyze_df["name"].isin(data_calls)]

# elif app_name in ["cm1"]:
#     df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
#     df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
#     df1['fhash'] = df1['fhash'].str.replace('.0', '', regex=False)  # Remove '.0'
#     df1['hhash'] = df1['hhash'].str.replace('.0', '', regex=False)  # Remove '.0'
#     df3 = analyzer.host_hash.reset_index()[['hash', 'name']] 
#     df3 = df3.rename(columns={'hash':'hhash'}) # update needed for this specific dataset
#     result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
#     result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
#     analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname']] 
#     analyze_df['id'] = analyze_df.index
#     analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")
#     data_calls = ['read', 'write', 'fread', 'fwrite', 'pread', 'pwrite']
#     data_df = analyze_df[analyze_df["name"].isin(data_calls)]
#     metadata_df = analyze_df[~analyze_df["name"].isin(data_calls)]




In [3]:
#####TEST CODE FROM HERE


# TEST FOR 3 Levels

In [47]:
import pandas as pd
import dask.dataframe as dd
import numpy as np

data = {
    'name':       ['write', 'read', 'write', 'read', 'write', 'read', 'write', 'read', 'write', 'read', 'write', 'read', 'write', 'read', 'write', 'read'],
    'cat':        ['A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B'],
    'size':       [100, 200, 150, 300, 250, 400, 120, 180, 300, 350, 400, 450, 500, 600, 700, 800],
    'ts':         [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16],
    'te':         [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17],
    'dur':        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    'trange':     [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    'mount_point':['/mnt1','/mnt1','/mnt2','/mnt2','/mnt1','/mnt1','/mnt3','/mnt3','/mnt4','/mnt4','/mnt5','/mnt5','/mnt6','/mnt6','/mnt7','/mnt7'],
    'hostname':   ['host1','host1','host1','host1','host2','host2','host2','host2','host3','host3','host3','host3','host3','host3','host3','host3'],
    'fhash':      ['fileA','fileA','fileB','fileB','fileC','fileC','fileD','fileD','fileE','fileE','fileF','fileF','fileG','fileG','fileH','fileH'],
    'pid':        [101, 102, 103, 104, 201, 202, 203, 203, 301, 302, 303, 304, 305, 305, 306, 307],
    'tid':        [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2],
    'prod':       [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0],
    'cons':       [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
}
# Convert to pandas DataFrame
pdf = pd.DataFrame(data)

# Convert to Dask DataFrame with 2 partitions
ddf = dd.from_pandas(pdf, npartitions=2)




In [58]:
wf = DFGrepWorkflow(ddf)

In [60]:
wf.select_events()

In [69]:
wf.get_wfGraph(level=1).compute()

,hostname,pid,fhash,ts
0,host3,301,file_same3,5
0,host1,101,file_same1,1
1,host2,201,file_same2,3


In [68]:
wf.get_wfGraph(level=2).compute()

,hostname,level_1,pid_prod,fhash,pid_cons,ts_prod,ts_cons
0,host3,0,302,file_cross_pid3,303,11,12
0,host1,0,102,file_cross_pid1,103,7,8
1,host2,0,202,file_cross_pid2,203,9,10


In [67]:
wf.get_wfGraph(level=3).compute()

,hostname_prod,fhash,hostname_cons,ts_prod,ts_cons
6,host1,file_cross_host1,host2,13,14
7,host2,file_cross_host2,host3,15,16
8,host3,file_cross_host3,host1,17,18
9,host1,file_cross_host4,host3,19,20
10,host1,file_multi1,host3,21,23
11,host1,file_multi1,host4,21,24
12,host2,file_multi1,host3,22,23
13,host2,file_multi1,host4,22,24
14,host1,file_multi2,host2,25,26
15,host1,file_multi2,host3,25,27
